# **Deep Natural Language Processing @ PoliTO**

---


**Teaching Assistant:** Giuseppe Gallipoli

**Credits:** Moreno La Quatra

**Practice 6:** Automatic Text Summarization - Extractive (part 1) and Abstractive (part 2) Summarization

# Automatic Text Summarization

Automatic text summarization is the task of producing a concise and fluent summary while preserving key information content and overall meaning. The summarization task is challenging because it requires a deep understanding of the text, including both its content and its style.

**Extractive Summarization** is the task of selecting a subset of the original text to form the summary. The selected sentences are concatenated to form the summary. Extractive summarization is the most common approach to text summarization as it only requires to understand the content of the text and estimate the importance of each sentence. It does not require to **generate** new text, which is a more complex task and requires both a deeper understanding of the text and text generation capabilities.

**Abstractive Summarization** is the task of generating a summary that is not a direct copy of the original text. It requires a deeper understanding of the text and the ability to generate new text. The model needs to understand the content of the text and the style of the author to produce a summary that is fluent and coherent with the original text.

## Extractive Text Summarization

The model needs to estimate the importance of each sentence in the text and select the most important sentences to form the summary.

![](https://images.deepai.org/machine-learning-models/8f66b1eb608e4eb681b2ec0c0631385c/summarization.jpg)

In this practice we will use the BBC News Summary dataset available in [Kaggle](https://www.kaggle.com/pariza/bbc-news-summary) to create the extractive summarization models. The following cell downloads the dataset and extracts it in the current directory.

In [1]:
%%capture
!wget https://github.com/MorenoLaQuatra/DeepNLP/raw/main/practices/P5/bbc_news.zip
!unzip bbc_news.zip

**Important note**

In some of the questions you are asked to use PageRank to estimate the importance of each sentence. However, the graph won't be directed and the PageRank algorithm may not converge. If this happens, you can use the following code to skip the PageRank algorithm and use a trivial importance score for each sentence.

```python
try:
    pr_scores = pagerank(G, max_iter=1000)
except Exception as e:
    print("The PageRank algorithm failed to converge. Returning the top sentences according to their position in the text.")
    pr_scores = {i: N-i for i in range(len(sentences))}
```

### **Question 1: Split data collection**

The data collection contains news articles belonging to different categories (e.g., business, sport, tech, etc.) and the corresponding summaries. In this question you will split the data collection into training, validation and test sets. The training set will be used to train the model, the validation set will be used to select the best model and the test set will be used to evaluate the final model. The goal is to stratify the data collection by category, so that each category is represented in the same proportion in each set. Be sure to select 10% of the data **of each category** for the test set. The remaining data can be split according to your preference.

**Note 1:** Some files can report UnicodeError, feel free to ignore it (`errors` parameter).
```python
f = open(FILENAME, 'r', encoding='utf-8', errors='ignore')
```

**Note 2:** You can fix encoding after file reading by using [ftfy](https://pypi.org/project/ftfy/) library.

```python
import ftfy
fixed_text = ftfy.fix_text(text)
```

The following cell installs the ftfy library that can be used to fix encoding issues.

In [2]:
!pip install ftfy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00


In [8]:
# your code here
import os
import pandas as pd
from sklearn.model_selection import train_test_split

base_path = "/content/BBC News Summary/News Articles"
summary_path = "/content/BBC News Summary/Summaries"
data = {'text': [], 'summary': [], 'category': []}

if os.path.exists(base_path):
    for cat in os.listdir(base_path):
        cat_path = os.path.join(base_path, cat)
        sum_cat_path = os.path.join(summary_path, cat)

        if os.path.isdir(cat_path):
            for file in os.listdir(cat_path):
                file_path = os.path.join(cat_path, file)
                sum_file_path = os.path.join(sum_cat_path, file)

                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    text = f.read()
                    data['text'].append(text)
                    data['category'].append(cat)
                    with open(sum_file_path, 'r', encoding='utf-8', errors='ignore') as s:
                        summary = s.read()
                        data['summary'].append(summary)

    df = pd.DataFrame(data)
    print(df.head())
else:
    print(f"The folder does not exists.")

train_val_df, test_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df['category']
)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=1/9,
    random_state=42,
    stratify=train_val_df['category']
)

                                                text  \
0  Smith aims to bring back respect\n\nScotland m...   
1  Liverpool revel in night of glory\n\nLiverpool...   
2  Serena becomes world number two\n\nSerena Will...   
3  Juninho demand for O'Neill talks\n\nJuninho's ...   
4  What now for Kelly Holmes?\n\nLast April, Kell...   

                                             summary category  
0  Smith has joined his first squad for a three-d...    sport  
1  Liverpool manager Rafael Benitez said their qu...    sport  
2  Her rise means Australia have a player in the ...    sport  
3  Hassel says Juninho, who has just bought a new...    sport  
4  With so much time spent in the spotlight, Holm...    sport  


### **Question 2: Unsupervised Text Summarization (TextRank algorithm)**

[TextRank](https://web.eecs.umich.edu/~mihalcea/papers/mihalcea.emnlp04.pdf) is an unsupervised text summarization approach that exploits graph-based ranking algorithms to estimate the importance of each sentence in the text. The algorithm is based on the following steps:

1. Each sentence is a node in a graph (undirected).
2. A pair of nodes is connected with an edge whose weight is computed according to the number of common words (see Note 1).
3. Once the graph is built, the importance of each sentence is estimated by computing the PageRank score of each node (you can use the [networkx](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.link_analysis.pagerank_alg.pagerank.html) library to compute the PageRank score).
4. The sentences are ranked according to their importance and the top ranked sentences are selected to form the summary.


Implement a `TextrankSummarizer` class that implements the TextRank algorithm. The class should have the following methods:

- `__init__(self, ...)`: the constructor of the class. You can add any parameter you want to the constructor if you think it is useful.
- `summarize(self, sentences, N)`: the method that computes the summary. The method takes as input the list of sentences `sentences` and the number of sentences `N` to select to form the summary. The method returns the list of selected sentences.
- Any other method you think it is useful. For example, you can add an internal method to compute the similarity between two sentences (see the example below).

**Note 1:** An example of the similarity function that can be used to compute graph weights is reported below. The function takes as input two sentences and returns the number of common words between the two sentences. The log function is used as a smoothing function to consider also the relative length of the sentences.

In [10]:
import math

def compute_similarity(tokens_sent_1, tokens_sent_2):
    n_common_words = len(set(tokens_sent_1) & set(tokens_sent_2))

    log_s1 = math.log10(len(tokens_sent_1))
    log_s2 = math.log10(len(tokens_sent_2))

    if log_s1 + log_s2 == 0:
        return 0

    return n_common_words / (log_s1 + log_s2)

In [11]:
# class skeleton
import networkx as nx

class TextrankSummarizer:

    def __init__(self):
        pass

    def summarize(self, sentences, N=2):
        # TODO: implement summarization
        G = nx.Graph()
        G.add_nodes_from(range(len(sentences)))
        for i in range(len(sentences)):
            for j in range(i+1, len(sentences)):
                weight = compute_similarity(sentences[i].split(), sentences[j].split())

                if weight > 0:
                    G.add_edge(i, j, weight=weight)

        try:
            pr_scores = nx.pagerank(G, weight='weight')
        except:
            pr_scores = {i: len(sentences)-i for i in range(len(sentences))}

        sorted_pr_scores = sorted(pr_scores.items(), key=lambda x: x[1], reverse=True)
        selected_sentences = [sentences[i] for i, _ in sorted_pr_scores[:N]]

        return selected_sentences

In [16]:
# your code here
import nltk

"""
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
"""

sample_text = test_df.iloc[0]['text']

sentences = sent_tokenize(sample_text)

summarizer = TextrankSummarizer()
summary = summarizer.summarize(sentences, N=2)

for sent in summary:
    print(f"- {sent}")

- But Hantuchova is in upbeat mood ahead of her clash with the younger Williams sister, who was handed a first-round bye.
- "It was really windy and I hadn't played in the wind.


### **Question 3: Unsupervised Text Summarization (TextRank + TF-IDF)**

In this question you will improve the TextRank algorithm by using the TF-IDF score of each sentence to compute the graph weights. Similarly to the previous question, the algorithm computes the importance of each sentence by computing the PageRank score of each node. The difference is that the graph weights are computed according to the TF-IDF score of each sentence.

Implement a `TextrankTFIDFSummarizer` class that contains the same methods of the `TextrankSummarizer` class. The only difference is that the graph weights are computed according to the TF-IDF score of each sentence.

Implement the class similarly to Q.2. This version uses a different similarity function to weigh edges connecting sentences. It uses [TF-IDF vectorization](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) and [cosine similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html) to compute sentence-to-sentence similarity.

The similarity function should take as input two sentences and compute the cosine similarity between the TF-IDF vectors of the two sentences. The similarity function can be implemented as a method of the `TextrankTFIDFSummarizer` class.

1. Compute TF-IDF vectors for each sentence.
2. Compute edges' weights using the cosine similarity between TF-IDF vector representations.

In [13]:
# your code here
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

class TextrankTFIDFSummarizer:

    def __init__(self):
        pass

    def summarize(self, sentences, N=2):
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(sentences)

        similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

        G = nx.Graph()
        G.add_nodes_from(range(len(sentences)))

        for i in range(len(sentences)):
            for j in range(i + 1, len(sentences)):
                weight = similarity_matrix[i][j]
                if weight > 0:
                    G.add_edge(i, j, weight=weight)

        try:
            scores = nx.pagerank(G, weight='weight')
        except:
            scores = {i: 0 for i in range(len(sentences))}

        ranked_sentences = sorted(((scores.get(i, 0), s) for i, s in enumerate(sentences)),
                                  key=lambda x: x[0], reverse=True)

        top_sentences = [s for score, s in ranked_sentences[:N]]

        return top_sentences

In [14]:
summarizer_tfidf = TextrankTFIDFSummarizer()
summary_tfidf = summarizer_tfidf.summarize(sentences, N=2)

for sent in summary_tfidf:
    print(f"- {sent}")

- But Hantuchova is in upbeat mood ahead of her clash with the younger Williams sister, who was handed a first-round bye.
- Hantuchova in Dubai last eight

Daniela Hantuchova moved into the quarter-finals of the Dubai Open, after beating Elene Likhotseva of Russia 7-5 6-4, and now faces Serena Williams.


### **Question 4: Unsupervised Text Summarization (Pretrained BERT)**

Both the TextRank and the TextRank + TF-IDF algorithms are based on the assumption that the importance of a sentence is related to the number of common words with other sentences. This assumption is not always true, especially when the sentences can express similar ideas using different words (e.g., synonyms). In this question you will use a pretrained BERT model to compute the **semantic similarity** between sentences. You can use `sentence-transformers` library to obtain sentence embeddings (https://www.sbert.net/) and compute the cosine similarity between the embeddings.

Implement a `BERTSummarizer` class that contains the same methods of the `TextrankSummarizer` class. The only difference is that the graph weights are computed according to the semantic similarity of each sentence.

Use `sentence-transformers` library to encode sentences into semantic-aware vectors and compute semantic similarity to connect sentences in the graph.

In [ ]:
# your code here
!pip install sentence-transformers

In [18]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

class BERTSummarizer:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)

    def summarize(self, sentences, N=2):
        embeddings = self.model.encode(sentences)

        similarity_matrix = cosine_similarity(embeddings)

        G = nx.Graph()
        G.add_nodes_from(range(len(sentences)))

        for i in range(len(sentences)):
            for j in range(i + 1, len(sentences)):
                weight = similarity_matrix[i][j]
                if weight > 0:
                    G.add_edge(i, j, weight=weight)

        try:
            scores = nx.pagerank(G, weight='weight')
        except:
            scores = {i: 0 for i in range(len(sentences))}

        ranked_sentences = sorted(((scores.get(i, 0), s) for i, s in enumerate(sentences)),
                                  key=lambda x: x[0], reverse=True)

        top_sentences = [s for score, s in ranked_sentences[:N]]

        return top_sentences

In [19]:
summarizer_bert = BERTSummarizer()
summary_bert = summarizer_bert.summarize(sentences, N=2)

for sent in summary_bert:
    print(f"- {sent}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

- "I feel I have an advantage (over Serena) because I have already played two matches on these courts," she said.
- Hantuchova in Dubai last eight

Daniela Hantuchova moved into the quarter-finals of the Dubai Open, after beating Elene Likhotseva of Russia 7-5 6-4, and now faces Serena Williams.


### **Question 5: ROUGE-based evaluation**

Automatic summarization models are usually evaluated using automatic metrics. The idea is to compare the automatically generated summary with the reference summary provided by humans. The most common metric used to evaluate automatic summarization models is [ROUGE](https://en.wikipedia.org/wiki/ROUGE_(metric)). The ROUGE metric is based on the idea that the automatically generated summary is considered correct if it contains an high number of n-grams that are also present in the reference summary.

Using only the **test set** obtained in Q.1, compare the performance of the three summarizers implemented in Q.2, Q.3 and Q.4.

Report their results in terms of average precision, recall and F1-score for ROUGE-2 metric. Set the number of extracted sentences to 4 for all summarizers.

**Which method obtains the best scores?**

**Note 1**: You can use the Python implementation of ROUGE available [here](https://pypi.org/project/rouge/).

In [20]:
!pip install rouge

In [23]:
# your code here
from rouge import Rouge
import numpy as np
from tqdm import tqdm
from nltk.tokenize import sent_tokenize

summ_base = TextrankSummarizer()
summ_tfidf = TextrankTFIDFSummarizer()
summ_bert = BERTSummarizer()

models = {
    'TextRank Base': summ_base,
    'TextRank TF-IDF': summ_tfidf,
    'TextRank BERT': summ_bert
}

results = {name: [] for name in models.keys()}
rouge = Rouge()

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), leave=False):
    text = row['text']
    reference_summary = row['summary']

    sentences = sent_tokenize(text)

    if len(sentences) < 4:
        continue

    for name, model in models.items():
        try:
            gen_sentences = model.summarize(sentences, N=4)
            generated_summary = " ".join(gen_sentences)

            scores = rouge.get_scores(generated_summary, reference_summary)[0]

            results[name].append(scores['rouge-2']['f'])

        except Exception as e:
            continue

print()
for name, scores in results.items():
    avg_score = np.mean(scores)
    print(f"{name}: {avg_score:.4f}")


TextRank Base: 0.5337
TextRank TF-IDF: 0.5459
TextRank BERT: 0.4935
